In [ ]:
import math
from collections import Counter, defaultdict


def makeLower(text):
    return text.lower().strip()

# def load_file(path):
#     with open(path, encoding="utf8") as f:
#         return makeLower(f.read())
    
def tokenize(sentence):
    # remove punctuations before splitting 
    token = makeLower(sentence).replace(".", " ").replace("!", " ").replace("?", " ").replace(";", " ").replace(":", " ").split()
    #print(["<s>"] + token + ["<e>"]) 
    return ["<s>"] + token + ["<e>"]

def bigram(ppSen):
    unknownCapacity = 0
    wordCounts = Counter()
    tokenizedSentence = []
    
    for line in ppSen:
        tokens = tokenize(line)
        tokenizedSentence.append(tokens)
        wordCounts.update(tokens) 

    vocab = set()  
    for w, c in wordCounts.items():
        # wordCounts.items() returns pairs: (word, count) so only add the word to the vocabulary
        # do not take into consideration OOV words 
        if w  in {"<s>", "<e>"} or c > unknownCapacity:
            vocab.add(w)

    vocab.add("<unk>")

    unigram = Counter()
    bigram = Counter()
    
    #calc the counts 
    for tokens in tokenizedSentence:
        updatedTokens = [w if w in vocab else "<unk>" for w in tokens]  # adding counts for <unk>
        unigram.update(updatedTokens) 
        for i in range(1, len(updatedTokens)):
            bigram[(updatedTokens[i-1], updatedTokens[i])] += 1
            
    return vocab, unigram, bigram

def laplace_bigram_prob(prev, word, uni_counts, bi_counts, V):
    prev_count = uni_counts.get(prev, 0)
    num = bi_counts.get((prev, word), 0) + 1    # also plus one here
    denom = prev_count + 1 * V                  # C(wi-1) + V abnd here 
    return num / denom

def logprob(sentence, vocab, uni_counts, bi_counts):
    tokens = tokenize(sentence)
    tokens = [w if w in vocab else "<unk>" for w in tokens]   
    V = len(vocab)
    logp = 0.0
    for i in range(1, len(tokens)):
        prev, w = tokens[i-1], tokens[i]
        p = laplace_bigram_prob(prev, w, uni_counts, bi_counts, V)
        logp += math.log(p)
    return logp

def predict(line, models):
    # my models structure is models: {lang: (vocab, uni_counts, bi_counts)}
    scores = {}
    for lang, (vocab, uni, bi) in models.items():
        scores[lang] = logprob(line, vocab, uni, bi)
    best = max(scores, key=scores.get)
    return best, scores

def load_lines(path):
    with open(path, encoding="utf8") as f:
        return [makeLower(l.rstrip("\n")) for l in f]

def write_predictions(predictions, out_path):
    with open(out_path, "w", encoding="utf8") as f:
        f.writelines(predictions)

def evaluate_predictions(predictedLines, solutionLines):
    sol = load_lines(solutionLines)
    correct = 0
    total = min(len(predictedLines), len(sol))
    for pred, g in zip(predictedLines, sol):
        if pred.strip() == g.strip():
            correct += 1
    return correct, total

# I added in like the special quotation marks that i know french has but i cant hit 100% accuracy so there might be some other characters that i missed
#load data to train 
en_text = load_lines("../Data/Input/LangId.train.English")
fr_text  = load_lines("../Data/Input/LangId.train.French")
it_text  = load_lines("../Data/Input/LangId.train.Italian")

#train on the traininng data 
en_model = bigram(en_text)
fr_model = bigram(fr_text)
it_model = bigram(it_text)

models = {
    "English": en_model,
    "French": fr_model,
    "Italian": it_model
}

with open("../Data/Validation/LangId.test", encoding="utf8") as f:
    test_lines = [l.rstrip("\n") for l in f]

output_lines = []

# Predict using bigram models numbering should match expected output
for i, line in enumerate(test_lines, 1): # starts from 1 
    line_id = str(i)
    # make it exactly the same way as training data
    sentence = line.lower().strip()
    pred, _ = predict(sentence, models)
    output_lines.append(f"{line_id} {pred}\n")

with open("../Data/Output/wordLangId.out", "w", encoding="utf8") as f:
    f.writelines(output_lines)

with open("../Data/Validation/labels.sol", encoding="utf8") as f:
    sol = f.readlines()

correct = 0
total = len(sol)

for mine, sol in zip(output_lines, sol):
    if mine.strip() == sol.strip():
        correct += 1
print("Laplace smoothing results:")
print(f"Correct: {correct}/{total}")
print(f"Accuracy: {correct/total:.2%}")



Laplace smoothing results:
Correct: 299/300
Accuracy: 99.67%


In [15]:
import math
from collections import Counter, defaultdict

def makeLower(text):
    return text.lower().strip()

def tokenize(sentence):
    punc_removed = makeLower(sentence).replace(".", " ").replace("!", " ").replace("?", " ").replace(";", " ").replace(":", " ")
    tokens = punc_removed.split()
    return ["<s>"] + tokens + ["<e>"]

def load_lines(path):
    with open(path, encoding="utf8") as f:
        return [makeLower(l.rstrip("\n")) for l in f]

def write_predictions(predictions, out_path):
    with open(out_path, "w", encoding="utf8") as f:
        f.writelines(predictions)

def evaluate_predictions(predictedLines, solutionLines):
    sol = load_lines(solutionLines)
    correct = 0
    total = min(len(predictedLines), len(sol))
    for pred, g in zip(predictedLines, sol):
        if pred.strip() == g.strip():
            correct += 1
    return correct, total

def bigram(ppSen):

    unknownCapacity = 0 
    wordCounts = Counter()
    tokenizedSentence = []
    
    for line in ppSen:
        tokens = tokenize(line)
        tokenizedSentence.append(tokens)
        wordCounts.update(tokens) 

    vocab = set()  
    for w, c in wordCounts.items():
        if w in {"<s>", "<e>"} or c > unknownCapacity:
            vocab.add(w)

    vocab.add("<unk>")

    unigram = Counter()
    bigram = Counter()
    
    for tokens in tokenizedSentence:
        updatedTokens = [w if w in vocab else "<unk>" for w in tokens] 
        unigram.update(updatedTokens) 
        for i in range(1, len(updatedTokens)):
            bigram[(updatedTokens[i-1], updatedTokens[i])] += 1
            
    return vocab, unigram, bigram

def noSmoothBigram(prev, word, uni_counts, bi_counts):
    prev_count = uni_counts.get(prev, 0)
    num = bi_counts.get((prev, word), 0) # C(w_i-1, w_i)
    
    # Denominator C(w_i-1)
    if prev_count == 0:
        # if a word was nebver seen, the probability is 0.
        return 0.0
    
    return num / prev_count # C(w_i-1, w_i) / C(w_i-1)

def logprob_noSmooth(sentence, vocab, uni_counts, bi_counts):

    tokens = tokenize(sentence)
    tokens = [w if w in vocab else "<unk>" for w in tokens]  
    logp = 0.0
    
    for i in range(1, len(tokens)):
        prev, w = tokens[i-1], tokens[i]
        
        p = noSmoothBigram(prev, w, uni_counts, bi_counts)
        if p == 0.0:
            # If the probability is zero, the log probability is -inf. penalizee i t 
            return -1e99 # -infitiivey score
        
        logp += math.log(p)
    return logp

def predict(line, models):
    scores = {}
    for lang, (vocab, uni, bi) in models.items():
        scores[lang] = logprob_noSmooth(line, vocab, uni, bi)
    
    best = max(scores, key=scores.get)
    return best, scores

en_text = load_lines("../Data/Input/LangId.train.English")
fr_text  = load_lines("../Data/Input/LangId.train.French")
it_text  = load_lines("../Data/Input/LangId.train.Italian")

en_model = bigram(en_text)
fr_model = bigram(fr_text)
it_model = bigram(it_text)

models = {
    "English": en_model,
    "French": fr_model,
    "Italian": it_model
}

with open("../Data/Validation/LangId.test", encoding="utf8") as f:
    test_lines = [l.rstrip("\n") for l in f]

output_lines = []

for i, line in enumerate(test_lines, 1): # starts from 1 
    line_id = str(i)
    # make it exactly the same way as training data
    sentence = line.lower().strip()
    pred, _ = predict(sentence, models)
    output_lines.append(f"{line_id} {pred}\n")

with open("../Data/Output/wordLangId.out", "w", encoding="utf8") as f:
    f.writelines(output_lines)

with open("../Data/Validation/labels.sol", encoding="utf8") as f:
    sol = f.readlines()

correct = 0
total = len(sol)

for mine, sol in zip(output_lines, sol):
    if mine.strip() == sol.strip():
        correct += 1
        
print("No Smoothing Bigram Model Results:")
print(f"Correct: {correct}/{total}")
print(f"Accuracy: {correct/total:.2%}")



No Smoothing Bigram Model Results:
Correct: 105/300
Accuracy: 35.00%


Correct: 299/300
Accuracy: 99.67%
